# Equilibrating a Liquid Against Atmospheric Gas — O₂, N₂, CO₂ Headspace

**The situation:** [usecase 01](01_predict_ph_simple_liquid.ipynb) solved
the KH₂PO₄ + NH₄Cl medium as a sealed liquid, no gas headspace. Leave that
same medium open to a typical atmosphere instead — simplified to O₂, N₂,
and CO₂ — and two things happen: CO₂ dissolves and joins the acid-base
chemistry already at play (it's a weak diprotic acid, same as phosphoric
acid was in usecase 01), while O₂ and N₂ dissolve too but take no part in
any reaction, so they only ever show up as a reported concentration, never
in the pH.

This extends usecase 01 with exactly one new mechanic — Henry's-law
gas-liquid partitioning — while staying at the same "one engine, one
`solve()`" level: the atmosphere is a fixed boundary condition (fixed
partial pressures), not a finite gas volume that the liquid could deplete,
so it converts to a dissolved-gas total the same way a weighed-out salt
does, and that total goes into `solve()` exactly like phosphate did in
usecase 01.

In [ ]:
import sys
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemistry.common_species import (
    H2O, H_plus, OH_minus,
    H3PO4, H2PO4_minus, HPO4_2minus, PO4_3minus,
    NH3, NH4_plus,
    CO2, HCO3_minus, CO3_2minus,
)
from PyOMES.chemistry.species import Species
from PyOMES.chemistry import HenryEquilibrium
from PyOMES.reactions.equilibrium import EquilibriumReaction
from PyOMES.reactions.stoichiometry import StoichiometryEntry
from PyOMES.chemical_equilibrium.nr_engine import NRChemicalEquilibriumEngine

def _e(sp, coeff, phase="liquid"):
    return StoichiometryEntry(species=sp, phase=phase, coefficient=coeff)

print("Imports OK")


## 1  Declare the chemistry

The phosphate and ammonium ladders are unchanged from usecase 01. Two more
reactions extend the acid-base network to cover dissolved CO₂ — carbonic
acid is diprotic, so it ladders down the same way phosphoric acid does,
just two steps instead of three:

| Reaction | log K | p$K_a$ | Role |
|---|---|---|---|
| H₂O ⇌ H⁺ + OH⁻ | −14.0 | 14.0 | water autoionization |
| H₃PO₄ ⇌ H₂PO₄⁻ + H⁺ | −2.15 | 2.15 | phosphate, 1st step |
| H₂PO₄⁻ ⇌ HPO₄²⁻ + H⁺ | −7.20 | 7.20 | phosphate, 2nd step |
| HPO₄²⁻ ⇌ PO₄³⁻ + H⁺ | −12.35 | 12.35 | phosphate, 3rd step |
| NH₄⁺ ⇌ NH₃ + H⁺ | −9.25 | 9.25 | ammonium/ammonia |
| CO₂(aq) + H₂O ⇌ HCO₃⁻ + H⁺ | −6.35 | 6.35 | carbonate, 1st step |
| HCO₃⁻ ⇌ CO₃²⁻ + H⁺ | −10.33 | 10.33 | carbonate, 2nd step |

`total_id="CO2"` on the two carbonate steps marks their own mass balance,
the same way `total_id="H3PO4"` did for phosphate — except this total
won't be weighed out as a salt, it'll be *computed* from an atmospheric
partial pressure via Henry's law in Section 2.

Alongside the acid-base network, each gas gets a Henry's-law solubility
constant (Sander convention: mol m⁻³ Pa⁻¹, plus a van 't Hoff temperature
sensitivity `dlnH`):

| Gas | H_ref (mol m⁻³ Pa⁻¹) | dlnH (K) | k_H at 25 °C (mol L⁻¹ atm⁻¹) | Reacts? |
|---|---|---|---|---|
| CO₂ | 3.4 × 10⁻⁴ | 2400 | 0.0345 | yes — carbonate ladder above |
| O₂  | 1.3 × 10⁻⁵ | 1500 | 0.00132 | no — inert here |
| N₂  | 6.4 × 10⁻⁶ | 1300 | 0.000648 | no — inert here |

O₂ and N₂ get no acid-base reaction at all (this notebook doesn't model
redox/respiration) — they're declared purely so their dissolved
concentration can be reported in Section 2, alongside the carbonate
chemistry that *does* feed back into pH.

In [ ]:
water = EquilibriumReaction(
    stoichiometry=[_e(H2O, -1), _e(H_plus, +1), _e(OH_minus, +1)],
    log_K=-14.0, label="water",
)
p1 = EquilibriumReaction(
    stoichiometry=[_e(H3PO4, -1), _e(H2PO4_minus, +1), _e(H_plus, +1)],
    log_K=-2.15, total_id="H3PO4", label="p1",
)
p2 = EquilibriumReaction(
    stoichiometry=[_e(H2PO4_minus, -1), _e(HPO4_2minus, +1), _e(H_plus, +1)],
    log_K=-7.20, total_id="H3PO4", label="p2",
)
p3 = EquilibriumReaction(
    stoichiometry=[_e(HPO4_2minus, -1), _e(PO4_3minus, +1), _e(H_plus, +1)],
    log_K=-12.35, total_id="H3PO4", label="p3",
)
nh4 = EquilibriumReaction(
    stoichiometry=[_e(NH4_plus, -1), _e(NH3, +1), _e(H_plus, +1)],
    log_K=-9.25, total_id="NH3", label="nh4",
)
co2_first = EquilibriumReaction(
    stoichiometry=[_e(CO2, -1), _e(H2O, -1), _e(HCO3_minus, +1), _e(H_plus, +1)],
    log_K=-6.35, total_id="CO2", label="co2_first",
)
co2_second = EquilibriumReaction(
    stoichiometry=[_e(HCO3_minus, -1), _e(CO3_2minus, +1), _e(H_plus, +1)],
    log_K=-10.33, total_id="CO2", label="co2_second",
)

# Henry's-law solubility constants. gas_species/liquid_species are left
# unset -- these are used only as a pCO2/pO2/pN2 -> dissolved-total
# converter (Section 2), never folded into the engine's own tableau, so
# they don't need to double as EquilibriumConstraint stoichiometry.
co2_henry = HenryEquilibrium(H_ref=3.4e-4, dlnH=2400.0, label="henry_CO2")
o2_henry = HenryEquilibrium(H_ref=1.3e-5, dlnH=1500.0, label="henry_O2")
n2_henry = HenryEquilibrium(H_ref=6.4e-6, dlnH=1300.0, label="henry_N2")

O2 = Species(id="O2", atoms={"O": 2}, charge=0)
N2 = Species(id="N2", atoms={"N": 2}, charge=0)

print("7 reactions declared, 3 Henry constants declared.")


## 2  Instantiate the equilibrium engine, and convert the atmosphere to totals

`engine` is a plain liquid-only `NRChemicalEquilibriumEngine`, same as
usecase 01 — the carbonate ladder makes it CO₂-aware, but nothing about
*how* the engine solves changes. The new step is converting a fixed
atmosphere (partial pressures) into the dissolved-gas totals `solve()`
expects, via Henry's law: $C^*_{aq} = k_H(T) \times p_{gas}$.

`kH_mol_L_atm()` below reproduces exactly the formula
`HenryEquilibrium` uses internally (Sander convention converted to
mol L⁻¹ atm⁻¹, van 't Hoff temperature correction) — written out
explicitly here rather than called from the class, the same way usecase
01 wrote out log K's explicitly rather than hiding them in a database
lookup.

In [ ]:
engine = NRChemicalEquilibriumEngine.from_reactions(
    [water, p1, p2, p3, nh4, co2_first, co2_second]
)

print("Masters:    ", engine.tableau.masters)
print("Secondaries:", [s.species_id for s in engine.tableau.secondaries])

def kH_mol_L_atm(H_ref, dlnH, T_K, T_ref=298.15):
    H_ref_mol_L_atm = (H_ref / 1000.0) * 101325.0
    return H_ref_mol_L_atm * math.exp(dlnH * (1.0 / T_K - 1.0 / T_ref))

T_K = 298.15  # 25 C

# Typical dry-air composition, simplified to O2/N2/CO2 per the notebook's
# scope (the remaining ~1% -- mostly argon -- is not modelled).
p_N2_atm = 0.78084
p_O2_atm = 0.20946
p_CO2_atm = 400e-6   # 400 ppm, present-day atmospheric CO2

CT_CO2_atm = kH_mol_L_atm(co2_henry.H_ref, co2_henry.dlnH, T_K) * p_CO2_atm
CT_O2_atm = kH_mol_L_atm(o2_henry.H_ref, o2_henry.dlnH, T_K) * p_O2_atm
CT_N2_atm = kH_mol_L_atm(n2_henry.H_ref, n2_henry.dlnH, T_K) * p_N2_atm

print(f"Dissolved CO2 at 400 ppm : {CT_CO2_atm*1e6:.3f} umol/L")
print(f"Dissolved O2  at 20.9%   : {CT_O2_atm*O2.MW*1e3:.3f} mg/L  "
      f"(compare: DO meters read ~8-9 mg/L for air-saturated water at 25 C)")
print(f"Dissolved N2  at 78.1%   : {CT_N2_atm*N2.MW*1e3:.3f} mg/L  "
      f"(compare: ~14-15 mg/L is the typical literature value)")


## 3  Does atmospheric CO₂ change the M9-like recipe's pH?

Same recipe as usecase 01 Section 3 — 22 mmol/L KH₂PO₄, 18.7 mmol/L
NH₄Cl — solved twice: sealed (`CO2` total = 0, usecase 01's answer) and
open to the atmosphere (`CO2` total = the value just computed).

In [ ]:
CT_P = 0.022   # mol/L KH2PO4 -> mol/L total phosphate, mol/L K+
CT_N = 0.0187  # mol/L NH4Cl  -> mol/L total ammoniacal N, mol/L Cl-

result_sealed = engine.solve(
    totals={"H3PO4": CT_P, "NH3": CT_N, "CO2": 0.0},
    strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
)
result_open = engine.solve(
    totals={"H3PO4": CT_P, "NH3": CT_N, "CO2": CT_CO2_atm},
    strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
)

print(f"pH, sealed (usecase 01)             = {result_sealed.pH:.4f}")
print(f"pH, open to atmosphere (400 ppm)    = {result_open.pH:.4f}")
print(f"Delta pH                            = {result_open.pH - result_sealed.pH:+.5f}")
print(f"[HCO3-] at equilibrium              = {result_open.species_mol_L['HCO3-']*1e6:.4f} umol/L")


The shift is tiny — micromolar CO₂ barely dents a medium buffered by
22 mmol/L phosphate. That's not a bug; Section 5 shows exactly where the
crossover into "CO₂ actually matters" sits.

## 4  Sanity check: pure water open to the atmosphere

With no phosphate buffer to mask it, dissolved atmospheric CO₂ alone
should reproduce a well-known textbook number: unpolluted rainwater sits
at about pH 5.6, set entirely by carbonic acid in equilibrium with
~400 ppm atmospheric CO₂ (see e.g. Stumm & Morgan, *Aquatic Chemistry*).
Reproducing that here checks the carbonate ladder declared in Section 1
against an independent, well-known reference point.

In [ ]:
result_pure = engine.solve(
    totals={"H3PO4": 0.0, "NH3": 0.0, "CO2": CT_CO2_atm},
    strong_ions={},
)
print(f"Pure water + 400 ppm atmospheric CO2 -> pH = {result_pure.pH:.3f}")
print("(textbook reference: unpolluted rainwater is ~pH 5.6, from atmospheric CO2 alone)")


## 5  Where atmospheric CO₂ actually matters: buffer capacity

Section 3 barely moved pH; Section 4 showed CO₂ alone sets pH ≈ 5.6.
Sweeping phosphate dose against headspace pCO₂ — from the present-day
atmosphere up to a CO₂-rich headspace like an anaerobic-digester biogas
space (tens of percent CO₂) — maps out the crossover between those two
regimes directly: flat contour lines (buffered, phosphate wins) versus
horizontal-banded ones (unbuffered, pCO₂ wins).

In [ ]:
n_grid = 30
pCO2_grid = np.logspace(np.log10(200e-6), 0.0, n_grid)   # 200 ppm - 100% CO2
CT_P_grid = np.logspace(-4, -1, n_grid)                    # 0.1 - 100 mmol/L KH2PO4

kH_CO2 = kH_mol_L_atm(co2_henry.H_ref, co2_henry.dlnH, T_K)

pH_grid = np.empty((n_grid, n_grid))
for i, p_co2 in enumerate(pCO2_grid):
    ct_co2 = kH_CO2 * p_co2
    for j, ct_p in enumerate(CT_P_grid):
        out = engine.solve(
            totals={"H3PO4": ct_p, "NH3": 0.0, "CO2": ct_co2},
            strong_ions={"CT_K": ct_p},
        )
        pH_grid[i, j] = out.pH

fig, ax = plt.subplots(figsize=(7.5, 6))
cf = ax.contourf(CT_P_grid * 1e3, pCO2_grid * 1e2, pH_grid, levels=20, cmap="viridis")
cs = ax.contour(CT_P_grid * 1e3, pCO2_grid * 1e2, pH_grid, levels=8,
                 colors="white", linewidths=0.6)
ax.clabel(cs, inline=True, fontsize=8, fmt="%.2f")

ax.scatter([22], [400e-6 * 100], marker="*", s=200, color="white",
           edgecolor="black", linewidth=0.8, zorder=5,
           label="Section 3 point (22 mM KH2PO4, 400 ppm)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("KH2PO4 (mmol/L)")
ax.set_ylabel("pCO2 (% atm)")
ax.set_title("pH vs. phosphate dose and headspace CO2\n(25 C, no ammonium)")
ax.legend(fontsize=8, loc="lower left")

cbar = fig.colorbar(cf, ax=ax)
cbar.set_label("pH")

plt.tight_layout()
plt.show()


## 6  Benchmark against PHREEQC

[Usecase 01 Section 6](01_predict_ph_simple_liquid.ipynb) works through
*why* an ideal engine and an activity-corrected one land at different
distances from PHREEQC in detail — same story applies here, so this
section just checks the carbonate chemistry added in this notebook holds
up under the same test: one parity plot, two series, each engine compared
against PHREEQC under a *matching* activity treatment (ideal vs. ideal,
Davies vs. WATEQ Debye-Hückel), swept across the atmospheric pCO₂ range
(200-5000 ppm) at the fixed M9-like point from Section 3.

As in usecase 01, this needs the optional `phreeqpython` package
(`pip install PyOMES[phreeqc]`); the cell below reports if it's missing
and the rest of the notebook is unaffected.

In [ ]:
try:
    import phreeqpython  # noqa: F401  -- actually probe for the optional dep;
    # PHREEQCChemicalEquilibriumEngine itself imports phreeqpython lazily
    # inside __init__, so importing only the wrapper class below would
    # succeed even without phreeqpython installed.
    from PyOMES.chemical_equilibrium.phreeqc_engine import PHREEQCChemicalEquilibriumEngine
    _HAVE_PHREEQC = True
    print("phreeqpython available - PHREEQC benchmark cell will run.")
except ImportError as exc:
    _HAVE_PHREEQC = False
    print(f"phreeqpython not installed ({exc}); skipping PHREEQC benchmark cell.")
    print("Install with: pip install PyOMES[phreeqc]")


In [ ]:
if _HAVE_PHREEQC:
    import tempfile
    from phreeqpython import PhreeqPython as _RawPhreeqPython

    engine_davies = NRChemicalEquilibriumEngine.from_reactions(
        [water, p1, p2, p3, nh4, co2_first, co2_second],
        use_activity=True, activity_model="davies",
    )

    # Near-ideal PHREEQC database: usecase 01 Section 6c's database,
    # extended with the carbonate master species and this notebook's own
    # log K's (-6.35, -10.33) so only solver mechanics can differ. Every
    # master species needs its own trivial "X = X" identity reaction (see
    # PO4-3/NH4+/K+/Cl- below) -- IPhreeqc silently fails the whole
    # database load without one (surfaces later as "No database is
    # loaded" on the first solve, not as a load-time error), so CO2 gets
    # one too.
    _IDEAL_DB_CO2 = """\
SOLUTION_MASTER_SPECIES
H       H+      -1.     H       1.008
H(0)    H2      0.0     H
H(1)    H+      -1.     0.0
E       e-      0.0     0.0     0.0
O       H2O     0.0     O       16.00
O(0)    O2      0.0     O
O(-2)   H2O     0.0     0.0
C       CO2     0.0     C       12.011
P       PO4-3   0.0     P       30.974
N       NH4+    0.0     N       14.0067
K       K+      0.0     K       39.098
Cl      Cl-     0.0     Cl      35.453

SOLUTION_SPECIES
H+ = H+
        log_k           0.0
        -gamma          1e6     0
e- = e-
        log_k           0.0
H2O = H2O
        log_k           0.0
2 H+ + 2 e- = H2
        log_k           -3.15
2 H2O = O2 + 4 H+ + 4 e-
        log_k           -86.08
H2O = OH- + H+
        log_k           -14.0
        -gamma          1e6     0
CO2 = CO2
        log_k           0.0
        -gamma          1e6     0
CO2 + H2O = HCO3- + H+
        log_k           -6.35
        -gamma          1e6     0
HCO3- = CO3-2 + H+
        log_k           -10.33
        -gamma          1e6     0
PO4-3 = PO4-3
        log_k           0.0
        -gamma          1e6     0
PO4-3 + H+ = HPO4-2
        log_k           12.35
        -gamma          1e6     0
PO4-3 + 2H+ = H2PO4-
        log_k           19.55
        -gamma          1e6     0
PO4-3 + 3H+ = H3PO4
        log_k           21.70
NH4+ = NH4+
        log_k           0.0
        -gamma          1e6     0
NH4+ = NH3 + H+
        log_k           -9.25
K+ = K+
        log_k           0.0
        -gamma          1e6     0
Cl- = Cl-
        log_k           0.0
        -gamma          1e6     0
END
"""
    _db_dir = Path(tempfile.gettempdir())
    (_db_dir / "vlsim_ideal_gas_liquid.dat").write_text(_IDEAL_DB_CO2)
    pp_ideal_gl = _RawPhreeqPython(database="vlsim_ideal_gas_liquid.dat",
                                    database_directory=_db_dir)

    def _solve_ideal_pq_co2(ct_c, ct_p=CT_P, ct_n=CT_N):
        sol = pp_ideal_gl.add_solution_raw({
            "C": ct_c * 1e3, "P": ct_p * 1e3, "N": ct_n * 1e3,
            "K": ct_p * 1e3, "Cl": ct_n * 1e3,
            "temp": 25.0, "pH": "7 charge", "units": "mmol/L",
        })
        ph = sol.pH
        sol.forget()
        return ph

    engine_pq_gl = PHREEQCChemicalEquilibriumEngine(
        {"H3PO4": CT_P * 1e3, "NH3": CT_N * 1e3, "CO2": 1.0,
         "CT_K": CT_P * 1e3, "CT_Cl": CT_N * 1e3},
        component_map={"H3PO4": "P", "NH3": "N(-3)", "CO2": "C",
                        "CT_K": "K", "CT_Cl": "Cl"},
        use_warmstart=False,
    )

    pCO2_vals_atm = np.logspace(np.log10(200e-6), np.log10(5000e-6), 20)  # 200-5000 ppm
    CT_CO2_vals = kH_CO2 * pCO2_vals_atm

    pH_ideal = np.array([
        engine.solve(totals={"H3PO4": CT_P, "NH3": CT_N, "CO2": ct_c},
                     strong_ions={"CT_K": CT_P, "CT_Cl": CT_N}).pH
        for ct_c in CT_CO2_vals
    ])
    pH_davies = np.array([
        engine_davies.solve(totals={"H3PO4": CT_P, "NH3": CT_N, "CO2": ct_c},
                             strong_ions={"CT_K": CT_P, "CT_Cl": CT_N}).pH
        for ct_c in CT_CO2_vals
    ])
    pH_pq_ideal = np.array([_solve_ideal_pq_co2(ct_c) for ct_c in CT_CO2_vals])
    pH_pq_default = np.array([
        engine_pq_gl.solve(totals={"H3PO4": CT_P, "NH3": CT_N, "CO2": ct_c,
                                    "CT_K": CT_P, "CT_Cl": CT_N}).pH
        for ct_c in CT_CO2_vals
    ])

    fig, ax = plt.subplots(figsize=(6, 6))
    all_vals = np.concatenate([pH_ideal, pH_davies, pH_pq_ideal, pH_pq_default])
    lo, hi = all_vals.min(), all_vals.max()
    pad = (hi - lo) * 0.08
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, zorder=0, label="1:1")

    ax.scatter(pH_pq_ideal, pH_ideal, s=32, color="tab:red", alpha=0.85,
               label="Ideal: NR (ideal) vs. PHREEQC (gamma -> 1)")
    ax.scatter(pH_pq_default, pH_davies, s=32, color="tab:green", marker="^", alpha=0.85,
               label="Nonideal: NR (Davies) vs. PHREEQC (WATEQ D-H)")

    ax.set_xlabel("PHREEQC pH")
    ax.set_ylabel("PyOMES pH")
    ax.set_title("Gas-liquid CO2 equilibration: parity vs. PHREEQC\n"
                 "(atmospheric pCO2 200-5000 ppm, fixed KH2PO4/NH4Cl)")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Max |ideal series|    (NR ideal  vs. PHREEQC gamma->1)  = {np.max(np.abs(pH_ideal - pH_pq_ideal)):.4f} pH units")
    print(f"Max |nonideal series| (NR Davies vs. PHREEQC WATEQ D-H) = {np.max(np.abs(pH_davies - pH_pq_default)):.4f} pH units")


## Where to go next

- **The activity-theory deep dive this notebook only summarizes** —
  [`01_predict_ph_simple_liquid.ipynb`](01_predict_ph_simple_liquid.ipynb)
  Section 6 walks through *why* ideal vs. Davies land at different
  distances from PHREEQC, and isolates solver-vs-solver agreement from
  activity-model disagreement.
- **A finite, sealed headspace and the time it actually takes to get
  there** — this notebook treats the atmosphere as an infinite reservoir
  at fixed partial pressure, reached instantaneously; a small sealed
  volume (e.g. a microplate well) is neither —
  [`02b_kinetic_co2_equilibration_microplate_well.ipynb`](02b_kinetic_co2_equilibration_microplate_well.ipynb)
  reruns this exact chemistry as a time-resolved, kLa-limited
  gas-liquid transfer and shows how much that changes the answer.
- **Wiring this into something that evolves over time** — a fermenter or
  reactor sparged continuously by a gas phase — see
  [`demos/model_api/chemistry/reaction_system.py`](../model_api/chemistry/reaction_system.py)
  and [`demos/builder/`](../builder/) for the full `Simulation` pattern.